# Cohorts, Retention and Funnels

## Quick notes
### What this covers
- Signup cohorts, the retention matrix (month 0, 1, 2, ...), churn, rolling retention, new vs returning vs resurrected users.
- Funnels: users per step, conversion and drop-off rates, ordered funnels, time between steps.
- Revenue per cohort (LTV), repeat purchase rate, DAU/WAU/MAU, and a simple A/B test table.
### Key points
- A cohort is a group of users who share a starting point (here: the month they signed up). Retention asks "of that group, how many came back N months later?"
- The denominator is always the cohort size, not "users active that month".
- SQLite has no date difference function. Months between two dates = `(year*12 + month)` of one minus the other.
- A funnel counts distinct users per step. A strict funnel also checks that the steps happened in the right order.
- Build small views (`user_months`, `cohorts`) once and reuse them: the main queries then stay short.
### Gotchas (the part worth revising twice)
- Recent cohorts have not had time to reach month 3. Showing `0%` there is wrong: it is "unknown" (`NULL`).
- `COUNT(*)` after joining users to orders counts orders, not users. Use `COUNT(DISTINCT user_id)`.
- Counting users per step without checking order lets users who skipped steps inflate later steps.
- Conversion "from the previous step" and "from the top" are different numbers. Say which one you show.
- SQLite cannot test statistical significance. It can compute rates and lift. Whether a difference is real needs a proper test, outside SQL.
- Month arithmetic with `date(x, '+1 month')` can overflow ("Jan 31 + 1 month" lands in March). Work with month numbers or `'start of month'`.
### When it matters
- Product analytics, subscription businesses, marketing ("did the March campaign bring users who stay?").
- Checkout and onboarding funnels, experiment readouts.
### One-line recap example
- `SELECT cohort, month_n, COUNT(*) FROM user_months GROUP BY cohort, month_n;` means "for each signup month, how many users were active 0, 1, 2 ... months later".

## 01. Seed data
- `users`: 10 users who signed up in Jan to Mar 2025. `activity`: dates they used the product. Data ends on 2025-04-30 (the "as of" date).
- `orders`: purchases. `events`: a checkout funnel for 8 shoppers (view, cart, checkout, paid). `ab`: an experiment, built further below.
- `cfg` holds the as-of date so queries do not depend on today.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE users (user_id INTEGER PRIMARY KEY, signup_date TEXT NOT NULL);
INSERT INTO users VALUES
    (1,'2025-01-03'),(2,'2025-01-10'),(3,'2025-01-20'),(4,'2025-01-28'),
    (5,'2025-02-02'),(6,'2025-02-12'),(7,'2025-02-25'),
    (8,'2025-03-04'),(9,'2025-03-15'),(10,'2025-03-22');
CREATE TABLE activity (user_id INTEGER NOT NULL REFERENCES users, activity_date TEXT NOT NULL);
INSERT INTO activity VALUES
    (1,'2025-01-03'),(1,'2025-01-04'),(1,'2025-02-05'),(1,'2025-03-10'),(1,'2025-04-02'),
    (2,'2025-01-10'),(2,'2025-02-11'),
    (3,'2025-01-20'),(3,'2025-03-01'),
    (4,'2025-01-28'),
    (5,'2025-02-02'),(5,'2025-03-03'),(5,'2025-03-04'),(5,'2025-04-20'),
    (6,'2025-02-12'),(6,'2025-03-14'),
    (7,'2025-02-25'),
    (8,'2025-03-04'),(8,'2025-04-06'),
    (9,'2025-03-15'),
    (10,'2025-03-22'),(10,'2025-04-01'),(10,'2025-04-30');
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, user_id INTEGER NOT NULL REFERENCES users,
                     order_date TEXT NOT NULL, amount INTEGER NOT NULL);
INSERT INTO orders VALUES
    (1,1,'2025-01-05',50),(2,1,'2025-02-06',30),(3,1,'2025-03-12',40),
    (4,2,'2025-01-12',20),(5,3,'2025-01-22',60),
    (6,5,'2025-02-03',25),(7,5,'2025-03-05',25),(8,5,'2025-04-22',35),
    (9,6,'2025-02-14',45),(10,8,'2025-03-05',70),(11,8,'2025-04-07',20),(12,10,'2025-03-23',15);
CREATE TABLE cfg (as_of TEXT NOT NULL);
INSERT INTO cfg VALUES ('2025-04-30');
SELECT * FROM users ORDER BY user_id;

user_id | signup_date
--------+------------
1       | 2025-01-03 
2       | 2025-01-10 
3       | 2025-01-20 
4       | 2025-01-28 
5       | 2025-02-02 
6       | 2025-02-12 
7       | 2025-02-25 
8       | 2025-03-04 
9       | 2025-03-15 
10      | 2025-03-22 
(10 rows)



## 02. Signup cohorts by month
- A cohort label is the signup month: `strftime('%Y-%m', signup_date)`.
- Count users per cohort. This count is the denominator for every retention percentage.

In [3]:
%%sql
SELECT strftime('%Y-%m', signup_date) AS cohort, COUNT(*) AS size
FROM users GROUP BY cohort ORDER BY cohort;

cohort  | size
--------+-----
2025-01 | 4   
2025-02 | 3   
2025-03 | 3   
(3 rows)



## 03. Months since signup (and two helper views)
- Month index = `year * 12 + month`. The difference between two month indexes is the number of calendar months between them. Signup in Jan 31, activity on Feb 1: month 1 (calendar months, not 30-day periods).
- `user_months`: one row per user per active month, with `month_n` (0 = the signup month). `DISTINCT` removes repeat visits in the same month.
- `cohorts`: size per cohort and `max_n`, the highest month number we can observe by the as-of date.

In [4]:
%%sql
CREATE VIEW user_months AS
SELECT DISTINCT a.user_id,
       strftime('%Y-%m', u.signup_date) AS cohort,
       strftime('%Y-%m', a.activity_date) AS active_month,
       CAST(strftime('%Y', a.activity_date) AS INT) * 12 + CAST(strftime('%m', a.activity_date) AS INT) AS mi,
       (CAST(strftime('%Y', a.activity_date) AS INT) * 12 + CAST(strftime('%m', a.activity_date) AS INT))
     - (CAST(strftime('%Y', u.signup_date) AS INT) * 12 + CAST(strftime('%m', u.signup_date) AS INT)) AS month_n
FROM activity a JOIN users u ON u.user_id = a.user_id;
CREATE VIEW cohorts AS
SELECT strftime('%Y-%m', u.signup_date) AS cohort, COUNT(*) AS size,
       (CAST(strftime('%Y', c.as_of) AS INT) * 12 + CAST(strftime('%m', c.as_of) AS INT))
     - (CAST(strftime('%Y', MIN(u.signup_date)) AS INT) * 12 + CAST(strftime('%m', MIN(u.signup_date)) AS INT)) AS max_n
FROM users u, cfg c GROUP BY strftime('%Y-%m', u.signup_date);
SELECT * FROM cohorts ORDER BY cohort;
SELECT * FROM user_months WHERE user_id IN (1, 3) ORDER BY user_id, mi;

cohort  | size | max_n
--------+------+------
2025-01 | 4    | 3    
2025-02 | 3    | 2    
2025-03 | 3    | 1    
(3 rows)

user_id | cohort  | active_month | mi    | month_n
--------+---------+--------------+-------+--------
1       | 2025-01 | 2025-01      | 24301 | 0      
1       | 2025-01 | 2025-02      | 24302 | 1      
1       | 2025-01 | 2025-03      | 24303 | 2      
1       | 2025-01 | 2025-04      | 24304 | 3      
3       | 2025-01 | 2025-01      | 24301 | 0      
3       | 2025-01 | 2025-03      | 24303 | 2      
(6 rows)



- User 1 was active twice in January (Jan 3 and 4) but has one `user_months` row for January. User 3 skipped February: months 0 and 2 only.

## 04. Retention in long format
- One row per (cohort, month_n): how many cohort users were active, divided by the cohort size.
- Long format is easy to chart and easy to filter. Join `cohorts` for the denominator.

In [5]:
%%sql
SELECT m.cohort, m.month_n, COUNT(*) AS active_users, c.size,
       ROUND(100.0 * COUNT(*) / c.size, 1) AS retention_pct
FROM user_months m JOIN cohorts c ON c.cohort = m.cohort
GROUP BY m.cohort, m.month_n
ORDER BY m.cohort, m.month_n;

cohort  | month_n | active_users | size | retention_pct
--------+---------+--------------+------+--------------
2025-01 | 0       | 4            | 4    | 100.0        
2025-01 | 1       | 2            | 4    | 50.0         
2025-01 | 2       | 2            | 4    | 50.0         
2025-01 | 3       | 1            | 4    | 25.0         
2025-02 | 0       | 3            | 3    | 100.0        
2025-02 | 1       | 2            | 3    | 66.7         
2025-02 | 2       | 1            | 3    | 33.3         
2025-03 | 0       | 3            | 3    | 100.0        
2025-03 | 1       | 2            | 3    | 66.7         
(9 rows)



## 05. The retention matrix (wide)
- One row per cohort, one column per month. SQLite has no `PIVOT`, so use `COUNT(*) FILTER (WHERE month_n = k)` per column (or `SUM(CASE ...)`).
- Month 0 is 100% here because every user was active in the signup month. In real data it is usually 100% by definition.

In [6]:
%%sql
SELECT m.cohort, c.size,
       ROUND(100.0 * COUNT(*) FILTER (WHERE m.month_n = 0) / c.size) AS m0,
       ROUND(100.0 * COUNT(*) FILTER (WHERE m.month_n = 1) / c.size) AS m1,
       ROUND(100.0 * COUNT(*) FILTER (WHERE m.month_n = 2) / c.size) AS m2,
       ROUND(100.0 * COUNT(*) FILTER (WHERE m.month_n = 3) / c.size) AS m3
FROM user_months m JOIN cohorts c ON c.cohort = m.cohort
GROUP BY m.cohort ORDER BY m.cohort;

cohort  | size | m0    | m1   | m2   | m3  
--------+------+-------+------+------+-----
2025-01 | 4    | 100.0 | 50.0 | 50.0 | 25.0
2025-02 | 3    | 100.0 | 67.0 | 33.0 | 0.0 
2025-03 | 3    | 100.0 | 67.0 | 0.0  | 0.0 
(3 rows)



## 06. Gotcha: cells that are not known yet
- The March cohort can only be observed for months 0 and 1 (data ends in April). In the matrix above its `m2` and `m3` show `0`, which looks like "everyone left". It really means "not observed yet".
- Fix: compare the column with `max_n` and return `NULL` when the month is beyond the horizon.

In [7]:
%%sql
SELECT m.cohort, c.size,
       CASE WHEN c.max_n >= 0 THEN ROUND(100.0 * COUNT(*) FILTER (WHERE m.month_n = 0) / c.size) END AS m0,
       CASE WHEN c.max_n >= 1 THEN ROUND(100.0 * COUNT(*) FILTER (WHERE m.month_n = 1) / c.size) END AS m1,
       CASE WHEN c.max_n >= 2 THEN ROUND(100.0 * COUNT(*) FILTER (WHERE m.month_n = 2) / c.size) END AS m2,
       CASE WHEN c.max_n >= 3 THEN ROUND(100.0 * COUNT(*) FILTER (WHERE m.month_n = 3) / c.size) END AS m3
FROM user_months m JOIN cohorts c ON c.cohort = m.cohort
GROUP BY m.cohort ORDER BY m.cohort;

cohort  | size | m0    | m1   | m2   | m3  
--------+------+-------+------+------+-----
2025-01 | 4    | 100.0 | 50.0 | 50.0 | 25.0
2025-02 | 3    | 100.0 | 67.0 | 33.0 | NULL
2025-03 | 3    | 100.0 | 67.0 | NULL | NULL
(3 rows)



## 07. Churn per calendar month
- A user "churned" in month M if they were active in M but not in M+1.
- Self-join `user_months` to itself on `mi + 1` with a `LEFT JOIN`. No match means churned.
- Skip the last month: we cannot know yet whether April users will return in May.

In [8]:
%%sql
SELECT a.active_month, COUNT(*) AS active, COUNT(n.user_id) AS returned_next_month,
       COUNT(*) - COUNT(n.user_id) AS churned,
       ROUND(100.0 * (COUNT(*) - COUNT(n.user_id)) / COUNT(*), 1) AS churn_pct
FROM user_months a
LEFT JOIN user_months n ON n.user_id = a.user_id AND n.mi = a.mi + 1
WHERE a.active_month < (SELECT strftime('%Y-%m', as_of) FROM cfg)
GROUP BY a.active_month ORDER BY a.active_month;

active_month | active | returned_next_month | churned | churn_pct
-------------+--------+---------------------+---------+----------
2025-01      | 4      | 2                   | 2       | 50.0     
2025-02      | 5      | 3                   | 2       | 40.0     
2025-03      | 7      | 4                   | 3       | 42.9     
(3 rows)



## 08. Rolling (unbounded) retention
- Classic retention asks "active exactly in month N". Rolling retention asks "active in month N or later". It never goes up and forgives users who skip a month.
- Use `MAX(month_n)` per user, then count users whose latest month is at least N.
- User 3 skipped February but came back in March. Exact retention counts them in month 2 only. Rolling retention counts them in months 1 and 2.

In [9]:
%%sql
WITH last_seen AS (
    SELECT user_id, cohort, MAX(month_n) AS last_n FROM user_months GROUP BY user_id, cohort
)
SELECT l.cohort, c.size,
       COUNT(*) FILTER (WHERE l.last_n >= 1) AS rolling_m1,
       COUNT(*) FILTER (WHERE l.last_n >= 2) AS rolling_m2,
       COUNT(*) FILTER (WHERE l.last_n >= 3) AS rolling_m3
FROM last_seen l JOIN cohorts c ON c.cohort = l.cohort
GROUP BY l.cohort ORDER BY l.cohort;

cohort  | size | rolling_m1 | rolling_m2 | rolling_m3
--------+------+------------+------------+-----------
2025-01 | 4    | 3          | 2          | 1         
2025-02 | 3    | 2          | 1          | 0         
2025-03 | 3    | 2          | 0          | 0         
(3 rows)



## 09. New, retained and resurrected users
- For every calendar month, split active users into three groups:
- new: first month (`month_n = 0`). retained: not new, and also active last month. resurrected: not new, and not active last month.
- `EXISTS` on the previous month does the check.

In [10]:
%%sql
SELECT m.active_month,
       COUNT(*) FILTER (WHERE m.month_n = 0) AS new_users,
       COUNT(*) FILTER (WHERE m.month_n > 0 AND EXISTS
           (SELECT 1 FROM user_months p WHERE p.user_id = m.user_id AND p.mi = m.mi - 1)) AS retained,
       COUNT(*) FILTER (WHERE m.month_n > 0 AND NOT EXISTS
           (SELECT 1 FROM user_months p WHERE p.user_id = m.user_id AND p.mi = m.mi - 1)) AS resurrected,
       COUNT(*) AS total_active
FROM user_months m
GROUP BY m.active_month ORDER BY m.active_month;

active_month | new_users | retained | resurrected | total_active
-------------+-----------+----------+-------------+-------------
2025-01      | 4         | 0        | 0           | 4           
2025-02      | 3         | 2        | 0           | 5           
2025-03      | 3         | 3        | 1           | 7           
2025-04      | 0         | 4        | 0           | 4           
(4 rows)



## 10. DAU, WAU, MAU and stickiness
- DAU, WAU, MAU = distinct active users per day, week, month. Always `COUNT(DISTINCT user_id)`: one user can have many events.
- Weekly: group by the Monday of the week, `date(d, '-6 days', 'weekday 1')` (see 04-Patterns/04-Time-Series-Patterns).
- Stickiness = average DAU divided by MAU. A value near 1 means everyone comes every day. Here it is tiny because the toy data is sparse.

In [11]:
%%sql
SELECT strftime('%Y-%m', activity_date) AS month,
       COUNT(DISTINCT user_id) AS mau,
       COUNT(*) AS events,
       ROUND(COUNT(DISTINCT user_id || activity_date) * 1.0 / 31, 3) AS avg_dau_if_31_days
FROM activity WHERE activity_date < '2025-02-01'
GROUP BY month;
SELECT date(activity_date, '-6 days', 'weekday 1') AS week_start, COUNT(DISTINCT user_id) AS wau
FROM activity WHERE activity_date < '2025-02-01'
GROUP BY week_start ORDER BY week_start;

month   | mau | events | avg_dau_if_31_days
--------+-----+--------+-------------------
2025-01 | 4   | 5      | 0.161             
(1 row)

week_start | wau
-----------+----
2024-12-30 | 1  
2025-01-06 | 1  
2025-01-20 | 1  
2025-01-27 | 1  
(4 rows)



- Rolling 30-day active users (a "sliding MAU") needs a join: for each reference date, count users with any activity in the 30 days up to and including it.

In [12]:
%%sql
WITH ref(day) AS (VALUES ('2025-01-31'), ('2025-02-28'), ('2025-03-31'), ('2025-04-30'))
SELECT r.day, COUNT(DISTINCT a.user_id) AS active_last_30_days
FROM ref r
LEFT JOIN activity a ON a.activity_date > date(r.day, '-30 days') AND a.activity_date <= r.day
GROUP BY r.day ORDER BY r.day;

day        | active_last_30_days
-----------+--------------------
2025-01-31 | 4                  
2025-02-28 | 5                  
2025-03-31 | 6                  
2025-04-30 | 4                  
(4 rows)



## 11. Funnel steps: users per step, conversion and drop-off
- `events` holds checkout events. Each user can have repeated or skipped events.
- Step 1: count distinct users per step. Then compute conversion from the previous step (`LAG`) and from the first step (`FIRST_VALUE`).
- Drop-off = users lost between two steps. Keep an explicit `step_no` so the order is not alphabetical.

In [13]:
%%sql
CREATE TABLE events (user_id INTEGER NOT NULL, step TEXT NOT NULL, ts TEXT NOT NULL);
INSERT INTO events VALUES
    (101,'view','2025-03-01 10:00:00'),(101,'cart','2025-03-01 10:05:00'),
    (101,'checkout','2025-03-01 10:10:00'),(101,'paid','2025-03-01 10:12:00'),
    (102,'view','2025-03-01 11:00:00'),(102,'cart','2025-03-01 11:20:00'),(102,'checkout','2025-03-01 11:25:00'),
    (103,'view','2025-03-01 12:00:00'),(103,'cart','2025-03-01 12:30:00'),
    (104,'view','2025-03-01 13:00:00'),
    (105,'view','2025-03-02 09:00:00'),(105,'cart','2025-03-02 09:30:00'),
    (105,'checkout','2025-03-03 09:30:00'),(105,'paid','2025-03-03 09:45:00'),
    (106,'view','2025-03-02 10:00:00'),(106,'view','2025-03-02 10:01:00'),(106,'cart','2025-03-02 10:10:00'),
    (107,'view','2025-03-02 11:00:00'),(107,'paid','2025-03-02 11:30:00'),
    (108,'view','2025-03-02 12:00:00'),(108,'checkout','2025-03-02 12:05:00'),
    (108,'cart','2025-03-02 12:10:00'),(108,'paid','2025-03-02 12:20:00');
CREATE TABLE funnel_steps (step_no INTEGER PRIMARY KEY, step TEXT NOT NULL);
INSERT INTO funnel_steps VALUES (1,'view'),(2,'cart'),(3,'checkout'),(4,'paid');
WITH per_step AS (
    SELECT f.step_no, f.step, COUNT(DISTINCT e.user_id) AS users
    FROM funnel_steps f LEFT JOIN events e ON e.step = f.step
    GROUP BY f.step_no, f.step
)
SELECT step, users,
       ROUND(100.0 * users / LAG(users) OVER (ORDER BY step_no), 1) AS pct_of_previous,
       ROUND(100.0 * users / FIRST_VALUE(users) OVER (ORDER BY step_no), 1) AS pct_of_top,
       LAG(users) OVER (ORDER BY step_no) - users AS dropped_off
FROM per_step ORDER BY step_no;

step     | users | pct_of_previous | pct_of_top | dropped_off
---------+-------+-----------------+------------+------------
view     | 8     | NULL            | 100.0      | NULL       
cart     | 6     | 75.0            | 75.0       | 2          
checkout | 4     | 66.7            | 50.0       | 2          
paid     | 4     | 100.0           | 50.0       | 0          
(4 rows)



- Look at `paid`: 4 users, the same as `checkout`. That looks like 100% conversion from checkout, but it is not true. User 107 paid without a cart or checkout event, and user 108's events are out of order. The next section fixes this.

## 12. Ordered funnel per user
- A strict funnel counts a user at a step only if they passed the previous step first, and the step happened at or after the previous one.
- One row per user with the first time of each step (`MIN(ts) FILTER (WHERE step = ...)`). Text timestamps in ISO format compare correctly.
- Then chain the checks: reached cart = cart time is not before view time; reached checkout = reached cart and checkout time is not before cart time, and so on.
- Comparing with `NULL` gives `NULL`, which is treated as "not reached". That is the behaviour we want here.

In [14]:
%%sql
CREATE VIEW user_funnel AS
WITH t AS (
    SELECT user_id,
           MIN(ts) FILTER (WHERE step = 'view')     AS t_view,
           MIN(ts) FILTER (WHERE step = 'cart')     AS t_cart,
           MIN(ts) FILTER (WHERE step = 'checkout') AS t_checkout,
           MIN(ts) FILTER (WHERE step = 'paid')     AS t_paid
    FROM events GROUP BY user_id
), r AS (
    SELECT *, (t_view IS NOT NULL) AS s1,
           (t_cart >= t_view) AS s2
    FROM t
), r3 AS (
    SELECT *, (s2 AND t_checkout >= t_cart) AS s3 FROM r
)
SELECT user_id, t_view, t_cart, t_checkout, t_paid, s1, COALESCE(s2, 0) AS s2, COALESCE(s3, 0) AS s3,
       COALESCE(s3 AND t_paid >= t_checkout, 0) AS s4
FROM r3;
SELECT user_id, s1, s2, s3, s4 FROM user_funnel ORDER BY user_id;

user_id | s1 | s2 | s3 | s4
--------+----+----+----+---
101     | 1  | 1  | 1  | 1 
102     | 1  | 1  | 1  | 0 
103     | 1  | 1  | 0  | 0 
104     | 1  | 0  | 0  | 0 
105     | 1  | 1  | 1  | 1 
106     | 1  | 1  | 0  | 0 
107     | 1  | 0  | 0  | 0 
108     | 1  | 1  | 0  | 0 
(8 rows)



In [15]:
%%sql
SELECT 'view' AS step, SUM(s1) AS users, 1 AS ord FROM user_funnel
UNION ALL SELECT 'cart', SUM(s2), 2 FROM user_funnel
UNION ALL SELECT 'checkout', SUM(s3), 3 FROM user_funnel
UNION ALL SELECT 'paid', SUM(s4), 4 FROM user_funnel
ORDER BY ord;

step     | users | ord
---------+-------+----
view     | 8     | 1  
cart     | 6     | 2  
checkout | 3     | 3  
paid     | 2     | 4  
(4 rows)



- Strict result: 8, 6, 3, 2. The loose count said 4 paid; strictly only users 101 and 105 completed the whole path in order. Users 107 (skipped) and 108 (checkout before cart) fall out at the point their sequence breaks.

## 13. Time between funnel steps
- For users who completed a step, the time to the next step is `(julianday(later) - julianday(earlier)) * 1440` minutes.
- Averages hide spread, so the second query shows min and max too. One slow step (user 105 took a day from cart to checkout, see the first table) would pull an average up a lot. SQLite has no `MEDIAN`.

In [16]:
%%sql
SELECT user_id,
       ROUND((julianday(t_cart) - julianday(t_view)) * 1440) AS view_to_cart_min,
       ROUND((julianday(t_checkout) - julianday(t_cart)) * 1440) AS cart_to_checkout_min,
       ROUND((julianday(t_paid) - julianday(t_checkout)) * 1440) AS checkout_to_paid_min
FROM user_funnel WHERE s4 = 1 ORDER BY user_id;
SELECT ROUND(AVG((julianday(t_cart) - julianday(t_view)) * 1440), 1) AS avg_view_to_cart_min,
       ROUND(MIN((julianday(t_cart) - julianday(t_view)) * 1440), 1) AS min_view_to_cart,
       ROUND(MAX((julianday(t_cart) - julianday(t_view)) * 1440), 1) AS max_view_to_cart
FROM user_funnel WHERE s2 = 1;

user_id | view_to_cart_min | cart_to_checkout_min | checkout_to_paid_min
--------+------------------+----------------------+---------------------
101     | 5.0              | 5.0                  | 2.0                 
105     | 30.0             | 1440.0               | 15.0                
(2 rows)

avg_view_to_cart_min | min_view_to_cart | max_view_to_cart
---------------------+------------------+-----------------
17.5                 | 5.0              | 30.0            
(1 row)



## 14. Repeat purchase rate
- Of the users who bought at least once, what share bought two or more times?
- Count orders per user first, then aggregate over that. Keep users with zero orders out of the denominator (or say so).

In [17]:
%%sql
WITH per_user AS (
    SELECT user_id, COUNT(*) AS n_orders FROM orders GROUP BY user_id
)
SELECT COUNT(*) AS buyers,
       COUNT(*) FILTER (WHERE n_orders >= 2) AS repeat_buyers,
       ROUND(100.0 * COUNT(*) FILTER (WHERE n_orders >= 2) / COUNT(*), 1) AS repeat_rate_pct,
       (SELECT COUNT(*) FROM users) AS all_users
FROM per_user;

buyers | repeat_buyers | repeat_rate_pct | all_users
-------+---------------+-----------------+----------
7      | 3             | 42.9            | 10       
(1 row)



## 15. Revenue and LTV by cohort
- Revenue per cohort per month since signup: join orders to users, compute `month_n` the same way as before.
- LTV so far = running revenue divided by the cohort size (not the number of buyers). Use a running `SUM ... OVER (PARTITION BY cohort ORDER BY month_n)`.
- Start from a grid of all (cohort, month_n) pairs that are observable, so months with no orders show their running value instead of vanishing.

In [18]:
%%sql
WITH rev AS (
    SELECT strftime('%Y-%m', u.signup_date) AS cohort,
           (CAST(strftime('%Y', o.order_date) AS INT) * 12 + CAST(strftime('%m', o.order_date) AS INT))
         - (CAST(strftime('%Y', u.signup_date) AS INT) * 12 + CAST(strftime('%m', u.signup_date) AS INT)) AS month_n,
           SUM(o.amount) AS revenue
    FROM orders o JOIN users u ON u.user_id = o.user_id
    GROUP BY cohort, month_n
), nums(month_n) AS (
    VALUES (0), (1), (2), (3)
), grid AS (
    SELECT c.cohort, c.size, n.month_n
    FROM cohorts c JOIN nums n ON n.month_n <= c.max_n
)
SELECT g.cohort, g.month_n, COALESCE(r.revenue, 0) AS revenue,
       SUM(COALESCE(r.revenue, 0)) OVER (PARTITION BY g.cohort ORDER BY g.month_n) AS cumulative_revenue,
       ROUND(1.0 * SUM(COALESCE(r.revenue, 0)) OVER (PARTITION BY g.cohort ORDER BY g.month_n) / g.size, 2) AS ltv_per_user
FROM grid g LEFT JOIN rev r ON r.cohort = g.cohort AND r.month_n = g.month_n
ORDER BY g.cohort, g.month_n;

cohort  | month_n | revenue | cumulative_revenue | ltv_per_user
--------+---------+---------+--------------------+-------------
2025-01 | 0       | 130     | 130                | 32.5        
2025-01 | 1       | 30      | 160                | 40.0        
2025-01 | 2       | 40      | 200                | 50.0        
2025-01 | 3       | 0       | 200                | 50.0        
2025-02 | 0       | 70      | 70                 | 23.33       
2025-02 | 1       | 25      | 95                 | 31.67       
2025-02 | 2       | 35      | 130                | 43.33       
2025-03 | 0       | 85      | 85                 | 28.33       
2025-03 | 1       | 20      | 105                | 35.0        
(9 rows)



## 16. A/B test comparison table
- Build 400 fake users with a recursive CTE: odd ids in variant B, even ids in variant A. Conversion is deterministic so results are repeatable.
- Report users, conversions, rate per variant, absolute difference (percentage points) and relative lift.
- SQL stops here. Whether 12.5% vs 10% is a real difference or noise needs a significance test, which belongs in Python or R. The squared z-score below shows the idea: values under 3.84 are not significant at the usual 95% level. A proper test also needs a planned sample size.

In [19]:
%%sql
CREATE TABLE ab (user_id INTEGER PRIMARY KEY, variant TEXT NOT NULL, converted INTEGER NOT NULL);
WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 400)
INSERT INTO ab
SELECT i, CASE WHEN i % 2 = 0 THEN 'A' ELSE 'B' END,
       CASE WHEN i % 2 = 0 THEN (i % 20 = 0) ELSE (i % 16 = 1) END
FROM n;
SELECT variant, COUNT(*) AS users, SUM(converted) AS conversions,
       ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conv_pct
FROM ab GROUP BY variant ORDER BY variant;

variant | users | conversions | conv_pct
--------+-------+-------------+---------
A       | 200   | 20          | 10.0    
B       | 200   | 25          | 12.5    
(2 rows)



In [20]:
%%sql
WITH v AS (
    SELECT variant, COUNT(*) AS n, SUM(converted) AS x, 1.0 * SUM(converted) / COUNT(*) AS p FROM ab GROUP BY variant
), pair AS (
    SELECT a.n AS na, a.x AS xa, a.p AS pa, b.n AS nb, b.x AS xb, b.p AS pb,
           1.0 * (a.x + b.x) / (a.n + b.n) AS pooled
    FROM v a, v b WHERE a.variant = 'A' AND b.variant = 'B'
)
SELECT ROUND(100 * pa, 2) AS rate_a_pct, ROUND(100 * pb, 2) AS rate_b_pct,
       ROUND(100 * (pb - pa), 2) AS diff_pp,
       ROUND(100 * (pb - pa) / pa, 1) AS relative_lift_pct,
       ROUND((pb - pa) * (pb - pa) / (pooled * (1 - pooled) * (1.0 / na + 1.0 / nb)), 2) AS z_squared
FROM pair;

rate_a_pct | rate_b_pct | diff_pp | relative_lift_pct | z_squared
-----------+------------+---------+-------------------+----------
10.0       | 12.5       | 2.5     | 25.0              | 0.63     
(1 row)



## 17. Bad vs good: counting users after a join
- Bad: `COUNT(*)` after joining users to orders counts order rows. A buyer with 3 orders is counted 3 times.
- Good: `COUNT(DISTINCT u.user_id)`. The same fan-out problem is explained in 02-Querying/01-Joins and 02-Querying/02-Aggregation-And-Grouping.
- Related trap: dividing by the users still active instead of the cohort size makes retention look better than it is, because the people who left disappear from the denominator.

In [21]:
%%sql
SELECT COUNT(*) AS wrong_count_rows,
       COUNT(DISTINCT u.user_id) AS right_count_users
FROM users u JOIN orders o ON o.user_id = u.user_id;

wrong_count_rows | right_count_users
-----------------+------------------
12               | 7                
(1 row)

